In [0]:
# Import libraries
from pyspark.sql.functions import *
from pyspark.sql import DataFrame
import logging
from datetime import datetime

In [0]:
# Logging cofiguration
logging.basicConfig(
    level = logging.INFO,
    format = "%(asctime)s %(levelname)s %(message)s"
)
logger = logging.getLogger("silver_pipeline")

In [0]:
# Configuration
PIPELINE_NAME = "silver_online_pipeline"
BRONZE_TABLE = "workspace.default.bronze_online_retail"
SILVER_TABLE = "workspace.default.silver_online_retail"

In [0]:
# Read source
def read_bronze() -> DataFrame:
    logger.info(f"Reading bronze table: {BRONZE_TABLE}")
    return (
        spark.table(BRONZE_TABLE)
    )
                

In [0]:
# Validation
def validate_source(df:DataFrame):
    logger.info("Running source validation")
    if df.limit(1).count() == 0:
        raise Exception(
            f"Source table: {BRONZE_TABLE} is empty"
            )


In [0]:
# Remove cancelled orders
def remove_cancelled_orders(df:DataFrame) -> DataFrame:
    logger.info("Removing cancelled orders")
    return (
        df.filter(~col("InvoiceNo").startswith("C"))
    )

In [0]:
# Remove Invalide Quqntity
def remove_invalid_quantity(df:DataFrame) -> DataFrame:
    logger.info("Removing invalid quantity")
    return (
        df.filter(col("Quantity") > 0)
    )

In [0]:
# Standardize Date
def convert_invoice_date(df:DataFrame) -> DataFrame:
    logger.info("Converting InvoiceDate")
    return (
        df.withColumn("InvoiceDate", try_to_timestamp(col("InvoiceDate"), lit("dd-MM-yyyy HH:mm")))
    )

In [0]:
# Validate dates
def validate_dates(df: DataFrame):
    invalid_dates = (
        df.filter(col("InvoiceDate").isNull()).count()
    )
    logger.info(f"Invalid dates found: {invalid_dates}")
    if invalid_dates > 0:
        raise Exception(
            f"{invalid_dates} invalid dates detected"
        )

In [0]:
# Business column
def add_sales_amount(df:DataFrame) -> DataFrame:
    logger.info("Adding sales amount")
    return (
        df.withColumn("SalesAmount", round(col("Quantity") * col("UnitPrice"), 2))
    )



In [0]:
# Metadata columns
def add_metadata(df: DataFrame) -> DataFrame:

    return (
        df
        .withColumn("load_timestamp",current_timestamp())
        .withColumn("pipeline_name",lit(PIPELINE_NAME))
    )

In [0]:
# Write to silver
def write_silver(df:DataFrame):
    logger.info(f"Writing silver table: {SILVER_TABLE}")
    return (
        df.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)
    )


In [0]:
# Audit Metrics
def log_pipeline_metrics(input_count,output_count,df):
    logger.info(f"Input Count: {input_count}")
    logger.info(f"Output Count: {output_count}")
    logger.info(f"Rejected Count: {input_count-output_count}")
    logger.info(
        f"Null CustomerID: "
        f"{df.filter(col('CustomerID').isNull()).count()}"
    )
    logger.info(
        f"Null Description: "
        f"{df.filter(col('Description').isNull()).count()}"
    )

In [0]:
# Main
def main():
    logger.info("Starting silver pipeline")
    bronze_df = read_bronze()
    validate_source(bronze_df)
    input_count = bronze_df.count()
    silver_df = (bronze_df
                 .transform(remove_cancelled_orders)
                 .transform(remove_invalid_quantity)
                 .transform(convert_invoice_date)
                 .transform(add_sales_amount)
                 .transform(add_metadata))
    
    output_count = silver_df.count()
    validate_dates(silver_df)
    log_pipeline_metrics(input_count,output_count,silver_df)
    write_silver(silver_df)
    logger.info("Finished silver pipeline")

In [0]:
# Entry point
try:
    main()
except Exception as e:
    logger.exception(f"Pipeline failed: {e}")
finally:
    logger.info("Silver Pipeline completed")

2026-06-04 20:20:13,317 INFO Starting silver pipeline
2026-06-04 20:20:13,318 INFO Reading bronze table: workspace.default.bronze_online_retail
2026-06-04 20:20:13,319 INFO Running source validation
2026-06-04 20:20:14,432 INFO Removing cancelled orders
2026-06-04 20:20:14,434 INFO Removing invalid quantity
2026-06-04 20:20:14,434 INFO Converting InvoiceDate
2026-06-04 20:20:14,435 INFO Adding sales amount
2026-06-04 20:20:16,022 INFO Invalid dates found: 0
2026-06-04 20:20:16,023 INFO Input Count: 541909
2026-06-04 20:20:16,023 INFO Output Count: 531285
2026-06-04 20:20:16,025 INFO Rejected Count: 10624
2026-06-04 20:20:16,849 INFO Null CustomerID: 133361
2026-06-04 20:20:17,497 INFO Null Description: 592
2026-06-04 20:20:17,498 INFO Writing silver table: workspace.default.silver_online_retail
2026-06-04 20:20:23,858 INFO Finished silver pipeline
2026-06-04 20:20:23,859 INFO Silver Pipeline completed
